Script for running all optimalizations that I have thusfar considered.

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go

snp = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CSPX ETF Stock Price History.csv")
china = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CNYA ETF Stock Price History.csv")
em = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "EIMI ETF Stock Price History.csv")
gold = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "XAD5 ETF Stock Price History.csv")
india = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "INR ETF Stock Price History.csv")
mscieurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "XMEU ETF Stock Price History.csv")
smallcapeurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "SXRJ ETF Stock Price History.csv")

dfs = [snp, china, em, gold, india, mscieurope, smallcapeurope]
date_sets = [set(df["Date"]) for df in dfs]

# Find the intersection of all date sets
common_dates = set.intersection(*date_sets)

# Filter each DataFrame to include only common dates
dfs = [df[df["Date"].isin(common_dates)] for df in dfs]

N = len(dfs)
T = len(dfs[0]) - 1

def correct_volume(vol_value):
    if isinstance(vol_value, (float, int)):
        return vol_value
    if vol_value[-1] == "K":
        return 1000 * float(vol_value[:-1])
    if vol_value[-1] == "M":
        return 1_000_000 * float(vol_value[:-1])
    return float(vol_value)

returns_all = np.empty((T, N))
sorted_returns_all = np.empty((T, N))
volumes_all = np.empty((T, N))
prices_all = np.empty((T, N))
dfs_new = []
for i, df in enumerate(dfs):
    df = df[::-1]
    df["Price"] = pd.to_numeric(df["Price"], errors="coerce")  # Handles str/NaN/object
    returns = df["Price"].pct_change().values[1:]
    returns_all[:, i] = returns
    sorted_returns_all[:, i] = sorted(returns)
    volumes_all[:, i] = df["Vol."].apply(correct_volume).values[1:]
    prices_all[:,i] = df["Price"].values[1:]
    dfs_new.append(df)

all_dates = dfs[0]["Date"].values[1:]
volumes_all = np.nan_to_num(volumes_all, nan=0.0)
dfs = [df.set_index("Date") for df in dfs]

In [9]:
def convert_format(int_repr):
    s = str(int_repr)
    year = s[0:4]
    month = s[4:6]
    day = s[6:8]
    return year + "-" + month + "-" + day

na_factors_daily = pd.read_csv(Path.cwd().parent / "data" / "FactorData" / "North_America_5_Factors_Daily.csv")
na_factors_daily = na_factors_daily.rename({"Unnamed: 0": "Date"}, axis=1).set_index("Date")
na_factors_daily.index = [convert_format(x) for x in na_factors_daily.index]

europe_factors_daily = pd.read_csv(Path.cwd().parent / "data" / "FactorData" / "Europe_5_Factors_Daily.csv")
europe_factors_daily = europe_factors_daily.rename({"Unnamed: 0": "Date"}, axis=1).set_index("Date")
europe_factors_daily.index = [convert_format(x) for x in europe_factors_daily.index]

asia_pacific_factors_daily = pd.read_csv(Path.cwd().parent / "data" / "FactorData" / "Asia_Pacific_ex_Japan_5_Factors_Daily.csv")
asia_pacific_factors_daily = asia_pacific_factors_daily.rename({"Unnamed: 0": "Date"}, axis=1).set_index("Date")
asia_pacific_factors_daily.index = [convert_format(x) for x in asia_pacific_factors_daily.index]

In [10]:
# date intersections
def match_dataframes(compare_df, to_transform_df):
    truth = [x in compare_df.index for x in to_transform_df.index]
    # also add returns
    return to_transform_df[truth]

# need to transform factor format from old to etf format
new_snp = match_dataframes(na_factors_daily, dfs[0])
new_china = match_dataframes(asia_pacific_factors_daily, dfs[1])
new_em = match_dataframes(asia_pacific_factors_daily, dfs[2])
new_gold = match_dataframes(europe_factors_daily, dfs[3])
new_india = match_dataframes(europe_factors_daily, dfs[4])
new_mscieurope = match_dataframes(europe_factors_daily, dfs[5])
new_smallcapeurope = match_dataframes(europe_factors_daily, dfs[6])
dfs = [new_snp, new_china, new_em, new_gold, new_india, new_mscieurope, new_smallcapeurope]
print(na_factors_daily.index)
na_factors_daily = match_dataframes(new_snp, na_factors_daily)
europe_factors_daily = match_dataframes(new_smallcapeurope, europe_factors_daily)
asia_pacific_factors_daily = match_dataframes(new_india, asia_pacific_factors_daily)

na_factors_daily = na_factors_daily.rename({old: old + "-NA" for old in na_factors_daily.columns}, axis=1)
europe_factors_daily = europe_factors_daily.rename({old: old + "-EU" for old in europe_factors_daily.columns}, axis=1)
asia_pacific_factors_daily = asia_pacific_factors_daily.rename({old: old + "-AS" for old in asia_pacific_factors_daily.columns}, axis=1)

factors_total = pd.concat([na_factors_daily, europe_factors_daily, asia_pacific_factors_daily], axis=1)
factors_total

Index(['1990-07-02', '1990-07-03', '1990-07-04', '1990-07-05', '1990-07-06',
       '1990-07-09', '1990-07-10', '1990-07-11', '1990-07-12', '1990-07-13',
       ...
       '2026-01-19', '2026-01-20', '2026-01-21', '2026-01-22', '2026-01-23',
       '2026-01-26', '2026-01-27', '2026-01-28', '2026-01-29', '2026-01-30'],
      dtype='str', length=9285)


,Mkt-RF-NA,SMB-NA,HML-NA,RMW-NA,CMA-NA,RF-NA,Mkt-RF-EU,SMB-EU,HML-EU,RMW-EU,CMA-EU,RF-EU,Mkt-RF-AS,SMB-AS,HML-AS,RMW-AS,CMA-AS,RF-AS
2015-04-14,0.17,-0.04,0.33,-0.03,0.13,0.00,0.39,0.39,0.06,0.13,-0.02,0.00,0.11,0.83,-0.30,-0.65,-0.40,0.00
2015-04-15,0.67,0.31,0.64,-0.26,-0.06,0.00,0.76,-0.08,0.41,-0.12,-0.03,0.00,-0.14,0.46,-0.06,-0.05,0.11,0.00
2015-04-16,-0.07,-0.05,-0.16,-0.15,-0.02,0.00,-0.05,0.35,-0.14,0.02,-0.10,0.00,1.11,0.72,-0.69,0.47,0.53,0.00
2015-04-17,-1.17,-0.35,0.23,0.08,0.09,0.00,-1.31,0.48,-0.21,0.03,-0.07,0.00,-0.57,0.69,0.35,-0.84,-0.18,0.00
2015-04-20,0.88,0.01,-0.16,0.40,-0.34,0.00,0.28,-0.34,0.17,-0.11,-0.05,0.00,-1.42,-0.18,0.59,-0.29,0.13,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-26,0.39,-0.67,0.01,0.65,0.10,0.01,0.48,-0.12,0.64,-0.16,0.30,0.01,0.28,0.02,0.58,-0.19,0.09,0.01
2026-01-27,0.37,-0.04,0.09,0.01,-0.45,0.01,1.89,-0.17,0.52,-0.02,0.05,0.01,1.90,-1.32,-0.17,0.73,-0.10,0.01
2026-01-28,-0.05,-0.25,0.31,0.02,-0.22,0.01,-1.46,0.73,0.86,-0.35,0.34,0.01,0.40,-0.23,0.84,0.01,-0.06,0.01
2026-01-29,-0.19,-0.13,1.45,0.87,0.97,0.01,-0.03,-0.24,0.57,0.66,0.44,0.01,-0.04,-0.60,1.03,0.22,0.32,0.01


In [ ]:
def correct_volume(vol_value):
    if isinstance(vol_value, (float, int)):
        return vol_value
    if vol_value[-1] == "K":
        return 1000 * float(vol_value[:-1])
    if vol_value[-1] == "M":
        return 1_000_000 * float(vol_value[:-1])
    return float(vol_value)

# construct returns matrix per asset

T, N = dfs[0].shape
returns = np.empty((T,N))
volumes = np.empty((T,N))
prices = np.empty((T,N))
dfs_new = []
for i, df in enumerate(dfs):
    df = df[::-1]
    returns[:,i] = df["Price"].pct_change()
    returns[0,i] = 0.0
    volumes[:,i] = df["Vol."].apply(correct_volume).values
    prices[:,i] = df["Price"].values
    dfs_new.append(df)
dfs = dfs_new

print('returns:'); print(returns)

graph = go.Figure()
for i in range(returns.shape[1]):
    L = list(range(T))
    cum_returns = np.cumprod(1 + returns[:,i]) - 1
    graph.add_trace(go.Scatter(x=list(range(len(cum_returns))), y=cum_returns))
graph.show()

Date
2026-01-30    255.06K
2026-01-29    190.69K
2026-01-28    105.14K
2026-01-27    248.45K
2026-01-26    142.65K
               ...   
2015-04-20    136.50K
2015-04-17    151.06K
2015-04-16    122.76K
2015-04-15     59.67K
2015-04-14    120.74K
Name: Vol., Length: 2492, dtype: str
returns:
[[ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.00583242 -0.00188324 -0.00078462 ...  0.00178147  0.00061637
   0.00415324]
 [ 0.00041419  0.03962264  0.01177856 ... -0.02489627 -0.00517433
  -0.01030928]
 ...
 [-0.00126856 -0.00335008  0.00426223 ... -0.00271318 -0.01166407
   0.00143021]
 [-0.01054912  0.00672269 -0.00727567 ...  0.00388651 -0.00246525
  -0.00728363]
 [ 0.00437814 -0.01001669 -0.00793974 ...  0.0058072   0.00736145
   0.00258956]]


In [12]:
def lower_part_mean(matrix):
    n, _ = matrix.shape
    s = 0.0
    for i in range(1,n):
        for j in range(i):
            s += matrix[i,j]
    N = (n-1)*n/2
    return s / N

def compute_statistics_rolling(returns, window_size, stepsize):

    all_corrs = []
    all_volas = []
    T, N = returns.shape
    w = np.ones(N) / N
    for idx in range(0, T - window_size, stepsize):
        section = returns[idx:idx+window_size,:]

        corr = np.corrcoef(section, rowvar=False) # (N,N)

        all_corrs.append(lower_part_mean(corr))
        returns_total = section @ w

        std = np.std(returns_total)
        all_volas.append(std)
    
    all_corrs = np.array(all_corrs)
    all_volas = np.array(all_volas)
    return all_corrs, all_volas


corrs, volas = compute_statistics_rolling(returns, 90, 30)

print('mean corrs:', np.mean(corrs))
print('mean volas:', np.mean(volas))
print('highest corrs:', np.max(corrs))
print('lowest corrs:', np.min(corrs))
print('lowest volas:', np.min(volas))
print('highest volas:', np.max(volas))
# results:
# based on this, determine threshold for correlation and volatility

vola_thresh = np.percentile(volas, 75)
corr_thresh = np.percentile(corrs, 75)

corrs_mean = np.mean(corrs)
corrs_std = np.std(corrs)
volas_mean = np.mean(volas)
volas_std = np.std(volas)

mean corrs: 0.3811465880837967
mean volas: 0.008024740190772008
highest corrs: 0.6358583075346117
lowest corrs: 0.18405877676544133
lowest volas: 0.003906785081833062
highest volas: 0.02187760923699238


In [ ]:
import tensorflow as tf
import numpy as np

alpha = 0.05   # tail probability for CVaR
lambda_ = 0.5  # CVaR penalty
learning_rate = 0.01
gamma = 0.08
num_steps = 2000
beta = 0.1 # penalty volatility
min_weight = 1/11 # 1/N = 1/7, so a little less
tau = 0.5 # to start
a = 1.0 # for stress environment tuning
b = 1.0 # for stress environment tuning
factor_risk_exposure = 0.4
risk_free = 0.03 # eg. bonds

def sigmoid(x):
    return 1/(1+np.exp(-x))

def normalize(x):
    return (x - tf.reduce_mean(x)) / (tf.math.reduce_std(x) + 1e-8)

def rank_transform(x):
    ranks = tf.argsort(tf.argsort(x)) # first argsort gives sorted indices, second gives rank
    return tf.cast(ranks, tf.float32) / tf.cast(tf.size(x), tf.float32)

def risk_parity_weights(train_data):
    # train_data does not yet include data for cash, so add this first
    print('RISK PARITY WEIGHTS PRINTOUT')
    print('SHAPE TRAIN DATA BEFORE:', train_data.shape)
    train_data_cash = np.zeros((train_data.shape[0], train_data.shape[1]+1))
    train_data_cash[:,:-1] = train_data.copy()
    train_data_cash[:,-1] = risk_free
    print('SHAPE TRAIN DATA AFTER:', train_data_cash.shape)
    # Compute volatilities for each asset
    volatilities = np.std(train_data_cash, axis=0)  # (N,)
    inverse_volatilities = 1.0 / (volatilities + 1e-8)  # Avoid division by zero

    # Normalize to get weights
    weights = inverse_volatilities / np.sum(inverse_volatilities)
    weights = np.clip(weights, a_min = min_weight, a_max = 1.0)
    print('RETURNIGN WEIGHTS OF SHAPE', weights.shape)
    return weights / np.sum(weights)

def gradient(returns, volume):

    # returns, volume: (T, N)
    # factor matrix: (9, T)
    # beta: (N, 9)
    T, N = returns.shape
    w = tf.Variable(tf.ones([N], dtype=tf.float32) / N)
    t = tf.Variable(0.0, dtype=tf.float32)  # VaR threshold

    optimizer = tf.keras.optimizers.Adam(learning_rate)
    turnover_values = []
    X_tf = tf.convert_to_tensor(returns, dtype=tf.float32) # (T,N)

    corr = np.corrcoef(returns, rowvar=False)
    mean_corr = lower_part_mean(corr)
    corr_standardized = (mean_corr - corrs_mean) / corrs_std

    w_prev = tf.identity(w)
    w_prev_normalized = tf.nn.softmax(w_prev)

    volume_tf = tf.convert_to_tensor(volume, dtype=tf.float32)

    for _ in range(num_steps):
        with tf.GradientTape() as tape:
            # enforce sum-to-1 via softmax
            w_normalized = tf.nn.softmax(w)
            w_normalized = tf.reshape(w_normalized, [-1, 1])

            portfolio_returns = tf.matmul(X_tf, tf.reshape(w_normalized, [-1,1])) # (T)
            losses = -portfolio_returns
            cvar = t + (1 / (1 - alpha)) * tf.reduce_mean(tf.nn.relu(losses - t))

            # build expected-return model
            # momentum per asset (N,)
            momentum = tf.reduce_mean(X_tf[-60:], axis=0)

            # volume per asset (N,)
            vol_short = tf.reduce_mean(volume_tf[-10:], axis=0)
            vol_long  = tf.reduce_mean(volume_tf[-60:], axis=0)
            volume_signal = (vol_short - vol_long) / (vol_long + 1e-8)

            volatility = tf.math.reduce_std(X_tf[-60:], axis=0)
            risk_adj_momentum = momentum / (volatility + 1e-8)

            m1 = normalize(momentum)
            m2 = normalize(risk_adj_momentum)
            m3 = normalize(volume_signal)

            # combine (N,)
            mu = 0.5 * m1 + 0.3 * m2 + 0.2 * m3
            mu = rank_transform(mu)

            turnover_penalty = tau * tf.reduce_sum(tf.square(w_normalized - w_prev_normalized))
            turnover = tf.reduce_sum(
                tf.abs(w_normalized - w_prev_normalized)
            ).numpy()
            turnover_values.append(turnover)

            weights_penalty = gamma * tf.reduce_sum(tf.math.square(w_normalized))
            expected_return = tf.tensordot(mu, w_normalized, axes=1)
            objective = -(expected_return - lambda_ * cvar) + weights_penalty + turnover_penalty

        # compute gradients
        grads = tape.gradient(objective, [w, t])
        optimizer.apply_gradients(zip(grads, [w, t]))

        w_prev_normalized = tf.identity(w_normalized)

    # after the optimization loop
    optimal_w = tf.nn.softmax(w).numpy()

    # compute volatility
    combined_returns = returns @ optimal_w # (T,)
    volatility = np.std(combined_returns)
    volatility_standardized = (volatility - volas_mean) / volas_std

    stress_score = sigmoid(a * corr_standardized + b * volatility_standardized)
    chosen_w = stress_score * np.ones(N) / N + (1 - stress_score) * optimal_w

    market_return = tf.reduce_mean(X_tf[-20:])   # avg across assets
    market_vol = tf.math.reduce_std(X_tf[-20:])
    crash_signal = tf.sigmoid(-5 * market_return + 5 * market_vol)
    cash_weight = crash_signal * 0.5  # up to 50% cash
    w_risky = (1 - cash_weight) * chosen_w
    w_final = tf.concat([w_risky, [cash_weight]], axis=0)

    if tf.reduce_any(w_final <= min_weight):
        w_final = np.clip(w_final, a_min = min_weight, a_max = 1.0)
        return w_final / np.sum(w_final), True, stress_score, np.mean(turnover_values)
    return w_final, False, stress_score, np.mean(turnover_values)

def gradient_factor(returns, factor_matrix, beta, volume):

    # returns, volume: (T, N)
    # factor matrix: (9, T)
    # beta: (N, 9)
    T, N = returns.shape
    w = tf.Variable(tf.ones([N], dtype=tf.float32) / N)
    t = tf.Variable(0.0, dtype=tf.float32)  # VaR threshold

    optimizer = tf.keras.optimizers.Adam(learning_rate)
    turnover_values = []
    X_tf = tf.convert_to_tensor(returns, dtype=tf.float32) # (T,N)

    corr = np.corrcoef(returns, rowvar=False)
    mean_corr = lower_part_mean(corr)
    corr_standardized = (mean_corr - corrs_mean) / corrs_std

    w_prev = tf.identity(w)
    w_prev_normalized = tf.nn.softmax(w_prev)

    factor_tf = tf.convert_to_tensor(factor_matrix, dtype=tf.float32)  # (9, T)
    beta_tf   = tf.convert_to_tensor(beta, dtype=tf.float32)          # (N, 9)
    volume_tf = tf.convert_to_tensor(volume, dtype=tf.float32)

    for _ in range(num_steps):
        with tf.GradientTape() as tape:
            # enforce sum-to-1 via softmax
            w_normalized = tf.nn.softmax(w)
            w_normalized = tf.reshape(w_normalized, [-1, 1])

            # risk model
            R_hat = tf.matmul(tf.transpose(factor_tf), tf.transpose(beta_tf)) # (T, N), predicted returns per factor
            portfolio_returns = (
                factor_risk_exposure * tf.matmul(R_hat, w_normalized)
                + (1 - factor_risk_exposure) * tf.matmul(X_tf, w_normalized)
            )
            cvar = t + (1 / (1 - alpha)) * tf.reduce_mean(tf.nn.relu(-portfolio_returns - t))

            # build expected-return model
            # momentum per asset (N,)
            momentum = tf.reduce_mean(X_tf[-60:], axis=0)

            # volume per asset (N,)
            vol_short = tf.reduce_mean(volume_tf[-10:], axis=0)
            vol_long  = tf.reduce_mean(volume_tf[-60:], axis=0)
            volume_signal = (vol_short - vol_long) / (vol_long + 1e-8)

            volatility = tf.math.reduce_std(X_tf[-60:], axis=0)
            risk_adj_momentum = momentum / (volatility + 1e-8)

            m1 = normalize(momentum)
            m2 = normalize(risk_adj_momentum)
            m3 = normalize(volume_signal)

            # combine (N,)
            mu = 0.5 * m1 + 0.3 * m2 + 0.2 * m3
            mu = rank_transform(mu)

            residuals = X_tf - R_hat
            residual_var = tf.reduce_mean(tf.square(residuals), axis=0)  # (N,)

            idio_risk = tf.reduce_sum(residual_var * tf.square(tf.squeeze(w_normalized)))

            turnover_penalty = tau * tf.reduce_sum(tf.square(w_normalized - w_prev_normalized))
            turnover = tf.reduce_sum(
                tf.abs(w_normalized - w_prev_normalized)
            ).numpy()
            turnover_values.append(turnover)

            weights_penalty = gamma * tf.reduce_sum(tf.math.square(w_normalized))
            expected_return = tf.tensordot(mu, w_normalized, axes=1)
            objective = -(expected_return - lambda_ * cvar) + weights_penalty + turnover_penalty + gamma*idio_risk

        # compute gradients
        grads = tape.gradient(objective, [w, t])
        optimizer.apply_gradients(zip(grads, [w, t]))

        w_prev_normalized = tf.identity(w_normalized)

    # after the optimization loop
    optimal_w = tf.nn.softmax(w).numpy()

    # compute volatility
    combined_returns = returns @ optimal_w # (T,)
    volatility = np.std(combined_returns)
    volatility_standardized = (volatility - volas_mean) / volas_std

    stress_score = sigmoid(a * corr_standardized + b * volatility_standardized)
    chosen_w = stress_score * np.ones(N) / N + (1 - stress_score) * optimal_w

    market_return = tf.reduce_mean(X_tf[-20:])   # avg across assets
    market_vol = tf.math.reduce_std(X_tf[-20:])
    crash_signal = tf.sigmoid(-5 * market_return + 5 * market_vol)
    cash_weight = crash_signal * 0.5  # up to 50% cash
    w_risky = (1 - cash_weight) * chosen_w
    w_final = tf.concat([w_risky, [cash_weight]], axis=0)

    if tf.reduce_any(w_final <= min_weight):
        w_final = np.clip(w_final, a_min = min_weight, a_max = 1.0)
        return w_final / np.sum(w_final), True, stress_score, np.mean(turnover_values)
    return w_final, False, stress_score, np.mean(turnover_values)

def max_drawdown(prices, weights):
    # prices (T,N)
    # weights (N,)
    T, N = prices.shape
    portfolio_values = prices @ weights # (T,)
    highest_peak = -1
    worst_pct = 0.0
    for t in range(T):
        highest_peak = max(highest_peak, portfolio_values[t])
        worst_pct = min(worst_pct, portfolio_values[t] / highest_peak - 1)
    return worst_pct

def perform_validation(w, validation_data, validation_prices):
    # validation data is shape (T, N)

    cash_returns = np.zeros((validation_data.shape[0], 1))  # or use risk-free rate
    validation_data_with_cash = np.hstack([validation_data, cash_returns])

    # Reshape w to (8, 1) for matrix multiplication
    w = tf.reshape(w, (-1, 1))

    # perform metrics
    total_return = validation_data_with_cash @ w  # shape (T, 1)
    total_return = total_return.numpy().flatten()  # convert to (T,)

    cumulative_returns = np.cumprod(total_return + 1) - 1
    std = np.std(total_return)
    downside_std = np.std(total_return[total_return < 0])
    sharpe = (255 * np.mean(total_return) - risk_free) / (np.sqrt(255) * std)
    sortino = (255 * np.mean(total_return) - risk_free) / (np.sqrt(255) * downside_std)

    alpha = 0.10

    var_alpha = np.quantile(total_return, alpha)
    cvar = total_return[total_return <= var_alpha].mean()
    mean_return = np.mean(total_return)

    md = max_drawdown(validation_prices, w[:-1].numpy())

    return var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return, md

In [16]:
from scipy import odr

def rolling_window(window_size, stepsize, validation_size, returns_data, volume_data, total_factor):
    # data is (T,N)

    sharpes_w_standard = []
    sharpes_w_eqw = []
    sharpes_w_rp = []
    sharpes_w_factor = []

    mean_return_w_standard = []
    mean_return_w_eqw = []
    mean_return_w_rp = []
    mean_return_w_factor = []

    sortinos_w_standard = []
    sortinos_w_eqw = []
    sortinos_w_rp = []
    sortinos_w_factor = []

    vars_w_standard = []
    vars_w_eqw = []
    vars_w_rp = []
    vars_w_factor = []

    cvars_w_standard = []
    cvars_w_eqw = []
    cvars_w_rp = []
    cvars_w_factor = []

    mds_w_standard = []
    mds_w_eqw = []
    mds_w_rp = []
    mds_w_factor = []

    T, N = returns_data.shape
    weights_standard_list = []
    weights_rp_list = []
    weights_factor_list = []

    stress_values_factor = []
    stress_values_std = []
    turnover_values_factor = []
    turnover_values_std = []

    beta =  np.empty((N, 9)) # beta values
    for idx in range(0, T - window_size - validation_size, stepsize):
        beta_prev = beta.copy()
        train_data_returns = returns_data[idx:idx+window_size]
        val_data_returns = returns_data[idx+window_size:idx+window_size+validation_size]
        train_data_volume = volume_data[idx:idx+window_size]
        val_data_prices = prices_all[idx+window_size:idx+window_size+validation_size]

        total_factor_slice = total_factor.iloc[idx:idx+window_size]

        factor_matrix = np.array([
            total_factor_slice["Mkt-RF-NA"].values,
            total_factor_slice["Mkt-RF-EU"].values,
            total_factor_slice["Mkt-RF-AS"].values,
            total_factor_slice["SMB-NA"].values,
            total_factor_slice["SMB-EU"].values,
            total_factor_slice["SMB-AS"].values,
            total_factor_slice["HML-NA"].values,
            total_factor_slice["HML-EU"].values,
            total_factor_slice["HML-AS"].values
        ])
        for i in range(N):
            # for each asset: run regression independently
            output = odr.ODR(odr.Data(factor_matrix, train_data_returns[:, i]), odr.multilinear).run()
            beta[i, :] = output.beta[1:] # skip constant term

        w_standard, cut_weights, stress_score, turnover_values = gradient(train_data_returns, train_data_volume)
        stress_values_std.append(stress_score)
        turnover_values_std.append(turnover_values)

        weights_standard_list.append(w_standard)
        w_factor, cut_weights, stress_score, turnover_values = gradient_factor(train_data_returns, factor_matrix, beta, train_data_volume)
        stress_values_factor.append(stress_score)
        turnover_values_factor.append(turnover_values)
        weights_factor_list.append(w_factor)
        w_rp = risk_parity_weights(train_data_returns)
        weights_rp_list.append(w_rp)

        (
            var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return, md
        ) = perform_validation(w_standard, val_data_returns, val_data_prices)

        sharpes_w_standard.append(sharpe)
        sortinos_w_standard.append(sortino)
        vars_w_standard.append(var_alpha)
        cvars_w_standard.append(cvar)
        mean_return_w_standard.append(mean_return)
        mds_w_standard.append(md)

        (
            var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return, md
        ) = perform_validation(np.ones(N+1) / (N+1), val_data_returns, val_data_prices)

        sharpes_w_eqw.append(sharpe)
        sortinos_w_eqw.append(sortino)
        vars_w_eqw.append(var_alpha)
        cvars_w_eqw.append(cvar)
        mean_return_w_eqw.append(mean_return)
        mds_w_eqw.append(md)        

        (
            var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return, md
        ) = perform_validation(w_rp, val_data_returns, val_data_prices)

        sharpes_w_rp.append(sharpe)
        sortinos_w_rp.append(sortino)
        vars_w_rp.append(var_alpha)
        cvars_w_rp.append(cvar)
        mean_return_w_rp.append(mean_return)
        mds_w_rp.append(md)

        (
            var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return, md
        ) = perform_validation(w_factor, val_data_returns, val_data_prices)

        sharpes_w_factor.append(sharpe)
        sortinos_w_factor.append(sortino)
        vars_w_factor.append(var_alpha)
        cvars_w_factor.append(cvar)
        mean_return_w_factor.append(mean_return)
        mds_w_factor.append(md)

    df = pd.DataFrame({
        "Sharpes": [np.mean(sharpes_w_eqw), np.mean(sharpes_w_standard), np.mean(sharpes_w_rp), np.mean(sharpes_w_factor)],
        "Sortinos": [np.mean(sortinos_w_eqw), np.mean(sortinos_w_standard), np.mean(sortinos_w_rp), np.mean(sortinos_w_factor)],
        "Vars": [np.mean(vars_w_eqw), np.mean(vars_w_standard), np.mean(vars_w_rp), np.mean(vars_w_factor)],
        "Cvars": [np.mean(cvars_w_eqw), np.mean(cvars_w_standard), np.mean(cvars_w_rp), np.mean(cvars_w_factor)],
        "MeanReturn": [np.mean(mean_return_w_eqw), np.mean(mean_return_w_standard), np.mean(mean_return_w_rp), np.mean(mean_return_w_factor)],
        "Highest maximum drawdowns": [np.min(mds_w_eqw), np.min(mds_w_standard), np.min(mds_w_rp), np.min(mds_w_factor)],
        "Mean maximum drawdowns": [np.mean(mds_w_eqw), np.mean(mds_w_standard), np.mean(mds_w_rp), np.mean(mds_w_factor)]
    }, index=["Equal weights", "Return + CVaR", "Risk Parity", "With factors"])

    weights_factor_list = np.array(weights_factor_list)
    weights_rp_list = np.array(weights_rp_list)
    weights_standard_list = np.array(weights_standard_list)

    return df, weights_factor_list, weights_rp_list, weights_standard_list, turnover_values_std, stress_values_std, turnover_values_factor, turnover_values_std

In [17]:
(
    df,
    weights_factor_list,
    weights_rp_list,
    weights_standard_list,
    turnover_values_std,
    stress_values_std,
    turnover_values_factor,
    turnover_values_std
) = rolling_window(
    window_size=252*2,
    stepsize=50,
    validation_size=252,
    returns_data=returns_all,
    volume_data=volumes_all,
    total_factor=factors_total
)
df

RISK PARITY WEIGHTS PRINTOUT
SHAPE TRAIN DATA BEFORE: (504, 7)
SHAPE TRAIN DATA AFTER: (504, 8)
RETURNIGN WEIGHTS OF SHAPE (8,)
RISK PARITY WEIGHTS PRINTOUT
SHAPE TRAIN DATA BEFORE: (504, 7)
SHAPE TRAIN DATA AFTER: (504, 8)
RETURNIGN WEIGHTS OF SHAPE (8,)
RISK PARITY WEIGHTS PRINTOUT
SHAPE TRAIN DATA BEFORE: (504, 7)
SHAPE TRAIN DATA AFTER: (504, 8)
RETURNIGN WEIGHTS OF SHAPE (8,)
RISK PARITY WEIGHTS PRINTOUT
SHAPE TRAIN DATA BEFORE: (504, 7)
SHAPE TRAIN DATA AFTER: (504, 8)
RETURNIGN WEIGHTS OF SHAPE (8,)
RISK PARITY WEIGHTS PRINTOUT
SHAPE TRAIN DATA BEFORE: (504, 7)
SHAPE TRAIN DATA AFTER: (504, 8)
RETURNIGN WEIGHTS OF SHAPE (8,)
RISK PARITY WEIGHTS PRINTOUT
SHAPE TRAIN DATA BEFORE: (504, 7)
SHAPE TRAIN DATA AFTER: (504, 8)
RETURNIGN WEIGHTS OF SHAPE (8,)
RISK PARITY WEIGHTS PRINTOUT
SHAPE TRAIN DATA BEFORE: (504, 7)
SHAPE TRAIN DATA AFTER: (504, 8)
RETURNIGN WEIGHTS OF SHAPE (8,)
RISK PARITY WEIGHTS PRINTOUT
SHAPE TRAIN DATA BEFORE: (504, 7)
SHAPE TRAIN DATA AFTER: (504, 8)
RETURNIG

,Sharpes,Sortinos,Vars,Cvars,MeanReturn,Highest maximum drawdowns,Mean maximum drawdowns
Equal weights,0.642599,0.875787,-0.007780,-0.013495,0.000373,-0.281273,-0.127660
Return + CVaR,0.728539,0.966650,-0.006512,-0.011341,0.000365,-0.292783,-0.125554
Risk Parity,0.293067,0.384390,-0.003458,-0.005998,0.000166,-0.281273,-0.127660
With factors,0.728557,0.966685,-0.006512,-0.011341,0.000365,-0.292782,-0.125554


In [23]:
mean_factor_weights = np.mean(weights_factor_list, axis=0)
mean_rp_weights = np.mean(weights_rp_list, axis=0)
mean_std_weights = np.mean(weights_standard_list, axis=0)
print(mean_factor_weights)
print(mean_rp_weights)
print(mean_std_weights)

from plotly.subplots import make_subplots
weights_fig = make_subplots(rows=1, cols=3)
num_steps = weights_factor_list.shape[0]

dfs = [new_snp, new_china, new_em, new_gold, new_india, new_mscieurope, new_smallcapeurope]

x = list(range(num_steps))
label_assets = ["SNP", "China", "EM", "Gold", "India", "MSCI-Europe", "SmallcapEurope"]

for i in range(len(mean_factor_weights)):
    for i, l in enumerate(label_assets):
        y = [item[i] for item in weights_factor_list]
        weights_fig.add_trace(go.Scatter(x=x, y=y, name=l), row=1, col=1)

for i in range(len(mean_rp_weights)):
    for i, l in enumerate(label_assets):
        y = [item[i] for item in weights_rp_list]
        weights_fig.add_trace(go.Scatter(x=x, y=y, name=l), row=1, col=2)

for i in range(len(mean_std_weights)):
    for i, l in enumerate(label_assets):
        y = [item[i] for item in weights_standard_list]
        weights_fig.add_trace(go.Scatter(x=x, y=y, name=l), row=1, col=3)

weights_fig.show()

[0.12375747 0.08672562 0.0911241  0.1728561  0.10770007 0.09540281
 0.09551316 0.22692055]
[0.05555577 0.05555577 0.05555577 0.05555577 0.05555577 0.05555577
 0.05555577 0.61110961]
[0.12375984 0.08672643 0.09112479 0.1728504  0.10770182 0.09540524
 0.09551208 0.22691947]


In [ ]:
def stress_test(df_data, start_train, end_train, end_valid):

    val_returns_total = []
    train_returns_total = []

    train_volume_total = []
    val_volume_total = []

    val_prices_total = []

    for df in df_data:
        df_datetime = df.copy()
        df_datetime["Date"] = pd.to_datetime(df_datetime["Date"])
        train = df_datetime[
            (df_datetime["Date"] >= pd.to_datetime(start_train))
            & (df_datetime["Date"] <= pd.to_datetime(end_train))]
        val = df_datetime[
            (df_datetime["Date"] >= pd.to_datetime(end_train))
            & (df_datetime["Date"] <= pd.to_datetime(end_valid))
        ]

        train_returns = train["Price"].pct_change().values[1:]
        val_returns = val["Price"].pct_change().values[1:]

        val_returns_total.append(val_returns)
        train_returns_total.append(train_returns)

        train_volume_total.append(np.nan_to_num(train["Vol_clean"].values[1:], nan=0.0))
        val_volume_total.append(np.nan_to_num(val["Vol_clean"].values[1:], nan=0.0))

        val_prices_total.append(val["Price"].values[1:])
        
    train_returns_total = np.array(train_returns_total).T # shape (num_days, N = num_assets)
    val_returns_total = np.array(val_returns_total).T
    train_volume_total = np.array(train_volume_total).T
    val_volume_total = np.array(val_volume_total).T
    val_prices_total = np.array(val_prices_total).T

    w_opt, cut_weights, stress_score, mean_turnover = gradient(train_returns_total, train_volume_total)
    
    return w_opt, perform_validation(w_opt, val_returns_total, val_prices_total), perform_validation(np.ones(N + 1) / (N+1), val_returns_total, val_prices_total), cut_weights, stress_score, mean_turnover


In [ ]:
start_train = "2019-10-01"
end_train = "2020-02-01"
end_valid = "2020-03-01"

chosen_weights, results_optimized, results_eqw, cut_weights, stress, turnover_values = stress_test(
    dfs_new, start_train, end_train, end_valid
)

var_alpha, cvar, sharpe, sortino, cumulative_returns_opt, mean_return, md = results_optimized

print('OPTIMIZED WEIGHTS RESULTS')
print('chosen weights:', chosen_weights)
print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)
print('maximum drawdown:', md)
print('capped weights:', cut_weights)
print()

var_alpha, cvar, sharpe, sortino, cumulative_returns_eqw, mean_return, md = results_eqw

print('EQUAL WEIGHTS RESULTS')
print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)
print('maximum drawdown:', md)
print()

print('turnover values:', turnover_values)
print('stress:', stress)

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(len(cumulative_returns_eqw))), y=cumulative_returns_eqw, name="equal weights"))
fig.add_trace(go.Scatter(x=list(range(len(cumulative_returns_opt))), y=cumulative_returns_opt, name="optimized weights"))
fig.show()


In [ ]:
start_train = "2021-02-22"
end_train = "2022-01-01"
end_valid = "2023-01-23"

chosen_weights, results_optimized, results_eqw, cut_weights, stress, turnover_values = stress_test(
    dfs_new, start_train, end_train, end_valid
)

var_alpha, cvar, sharpe, sortino, cumulative_returns_opt, mean_return, md = results_optimized

print('OPTIMIZED WEIGHTS RESULTS')
print('chosen weights:', chosen_weights)
print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)
print('maximum drawdown:', md)
print('capped weights:', cut_weights)
print()

var_alpha, cvar, sharpe, sortino, cumulative_returns_eqw, mean_return, md = results_eqw

print('EQUAL WEIGHTS RESULTS')
print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)
print('maximum drawdown:', md)
print()

print('turnover values:', turnover_values)
print('stress:', stress)

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(len(cumulative_returns_eqw))), y=cumulative_returns_eqw, name="equal weights"))
fig.add_trace(go.Scatter(x=list(range(len(cumulative_returns_opt))), y=cumulative_returns_opt, name="optimized weights"))
fig.show()


In [ ]:
start_train = "2024-07-01"
end_train = "2025-01-01"
end_valid = "2025-03-01"

chosen_weights, results_optimized, results_eqw, cut_weights, stress, turnover_values = stress_test(
    dfs_new, start_train, end_train, end_valid
)

var_alpha, cvar, sharpe, sortino, cumulative_returns_opt, mean_return, md = results_optimized

print('OPTIMIZED WEIGHTS RESULTS')
print('chosen weights:', chosen_weights)
print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)
print('maximum drawdown:', md)
print('capped weights:', cut_weights)
print()

var_alpha, cvar, sharpe, sortino, cumulative_returns_eqw, mean_return, md = results_eqw

print('EQUAL WEIGHTS RESULTS')
print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)
print('maximum drawdown:', md)
print()

print('turnover values:', turnover_values)
print('stress:', stress)

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(len(cumulative_returns_eqw))), y=cumulative_returns_eqw, name="equal weights"))
fig.add_trace(go.Scatter(x=list(range(len(cumulative_returns_opt))), y=cumulative_returns_opt, name="optimized weights"))
fig.show()
